In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
image_dir = f"{base_dir}/AntM2C_image"
preprocessed_dir = f"{base_dir}/AntM2C_preprocessed"
image_metadata_path = f"{preprocessed_dir}/image_paths.csv"

In [3]:
# Import libraries
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from PIL import Image

from transformers import AutoImageProcessor, AutoModel

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load Pretrained ViT Model

The `google/vit-base-patch16-224-in21k` model will be used from HuggingFace:

- The `AutoImageProcessor` handles resizing and normalization according to the model’s training config
- The `AutoModel` provides a 768-dim embedding via the `[CLS]` token
- The model is set to inference mode to reduce memory usage and disable gradient tracking

In [5]:
# Load pretrained ViT model
model_name = "google/vit-base-patch16-224-in21k"

# Load processor and model
image_processor = AutoImageProcessor.from_pretrained(model_name)
vit_model = AutoModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [6]:
# Inference mode
vit_model.eval()
vit_model.to(device)
print("ViT model loaded and ready on", device)

ViT model loaded and ready on cuda


In [7]:
# Check output dimension with dummy image
dummy_input = torch.rand(1, 3, 224, 224).to(device)
with torch.no_grad():
    output = vit_model(pixel_values=dummy_input)
    cls_embedding = output.last_hidden_state[:, 0, :]  # [CLS] token

print("Embedding shape:", cls_embedding.shape)

Embedding shape: torch.Size([1, 768])


In [ ]:
!git status